In [1]:
import numpy as np
import pandas as pd
import sklearn
import torch

In [2]:
df = pd.read_csv("Housing.csv")

In [3]:
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [27]:
from sklearn.model_selection import train_test_split


def func_train_test_val_split(X, y):
    """
    splits into train,test,val
    @return:  X_train,y_train,X_val,y_val,X_test,y_test
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, random_state=42, test_size=0.3
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=42
    )
    return X_train, y_train, X_val, y_val, X_test, y_test

In [23]:
df.dropna(inplace=True)

In [24]:
df["price"].dtype

dtype('int64')

In [32]:
X = df[["area", "bedrooms", "bathrooms", "parking"]]
y = df["price"]

In [162]:
y = np.log(y)

In [163]:
X_train, y_train, X_val, y_val, X_test, y_test = func_train_test_val_split(X, y)

In [164]:
X_val.shape

(39, 4)

In [165]:
type(X_train)

pandas.DataFrame

In [166]:
from sklearn.preprocessing import StandardScaler

In [167]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [168]:
X_test = scaler.transform(X_test)

In [169]:
X_train.dtype

dtype('float64')

In [170]:
X_train

array([[-0.52898161,  0.05702103, -0.56154655, -0.81839922],
       [-0.4400649 ,  0.05702103, -0.56154655, -0.81839922],
       [ 0.57877248,  0.05702103, -0.56154655,  1.5535714 ],
       ...,
       [ 0.18049969,  0.05702103, -0.56154655,  0.36758609],
       [-1.18566861,  0.05702103, -0.56154655, -0.81839922],
       [ 3.62787489,  0.05702103, -0.56154655, -0.81839922]],
      shape=(342, 4))

In [171]:
print(X_train)

[[-0.52898161  0.05702103 -0.56154655 -0.81839922]
 [-0.4400649   0.05702103 -0.56154655 -0.81839922]
 [ 0.57877248  0.05702103 -0.56154655  1.5535714 ]
 ...
 [ 0.18049969  0.05702103 -0.56154655  0.36758609]
 [-1.18566861  0.05702103 -0.56154655 -0.81839922]
 [ 3.62787489  0.05702103 -0.56154655 -0.81839922]]


In [114]:
type(X_train)

numpy.ndarray

## Linear regression with torch

In [115]:
X_train

array([[-0.52898161,  0.05702103, -0.56154655, -0.81839922],
       [-0.4400649 ,  0.05702103, -0.56154655, -0.81839922],
       [ 0.57877248,  0.05702103, -0.56154655,  1.5535714 ],
       ...,
       [ 0.18049969,  0.05702103, -0.56154655,  0.36758609],
       [-1.18566861,  0.05702103, -0.56154655, -0.81839922],
       [ 3.62787489,  0.05702103, -0.56154655, -0.81839922]],
      shape=(342, 4))

In [116]:
# mlp = torch.matmul(X_train,)

In [117]:
from torch import nn


class MLP(torch.nn.Module):
    def __init__(self, inp, output):
        super(MLP, self).__init__()
        self.input_layer = nn.Linear(inp, 4)

    def forward(self, x):
        return self.relu_stack(x)

In [172]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.float32)

In [56]:
mlp = MLP(X_train.shape[0], X_train.shape[1])

In [57]:
mlp

MLP(
  (relu_stack): Sequential(
    (0): Linear(in_features=342, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=342, out_features=4, bias=True)
  )
)

In [68]:
loss_fn = torch.nn.MSELoss()
optim = torch.optim.SGD(model.parameters(), lr=0.3)

In [72]:
X_train.shape[1]

4

In [185]:
epochs = 10
lr = 0.01

In [194]:
def make_mlp(
    num_inputs: int, num_outputs: int, activation: Any = None, hidden_sizes: int = [128]
) -> nn.Module:

    if activation == "relu":
        act_cls = nn.ReLU
    elif activation == "tanh":
        act_cls = nn.Tanh
    elif activation == "none" or not activation:
        act_cls = nn.Identity
    else:
        raise ValueError(f"Unexpected activation={repr(activation)}")

    net = [nn.Linear(num_inputs, hidden_sizes[0]), act_cls()]

    for i in range(1, len(hidden_sizes)):
        net.extend([nn.Linear(hidden_sizes[i - 1], hidden_sizes[i]), act_cls()])

    net.extend([nn.Linear(hidden_sizes[-1], num_outputs)])

    return nn.Sequential(*net)

In [186]:
hidden_unit = [8, 4]
#
from torch import nn

input_dim = X_train.shape[1]
hidden_units = [4]  # Easily add more layers here (e.g., [8, 4])
output_dim = 1

# 2. Build the model safely
model = nn.Sequential(
    nn.Linear(input_dim, hidden_units[0]),
    nn.ReLU(),
    nn.Linear(hidden_units[0], output_dim),
)
loss_fn = torch.nn.MSELoss()
optim = torch.optim.SGD(model.parameters(), lr=lr)

In [196]:
model2 = make_mlp(input_dim, output_dim, "relu", hidden_units)

In [197]:
model2

Sequential(
  (0): Linear(in_features=4, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)

In [198]:
model

Sequential(
  (0): Linear(in_features=4, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)

In [175]:
y_train

tensor([15.2994, 15.3904, 15.5630, 15.3904, 14.9517, 15.9725, 16.0238, 15.4737,
        15.2506, 15.0582, 15.2261, 15.6561, 14.9849, 15.3976, 15.3459, 14.8939,
        16.0466, 15.1726, 15.3758, 14.6220, 15.3459, 15.4737, 15.6835, 15.2253,
        15.2506, 15.3151, 14.7398, 15.7463, 14.4334, 15.9437, 15.6561, 15.6222,
        15.0481, 15.0378, 15.5383, 15.7714, 15.1056, 14.9517, 15.1993, 15.0683,
        15.2785, 15.6336, 16.1050, 15.0481, 15.0275, 15.9765, 14.7672, 15.0683,
        15.3609, 15.0582, 15.7910, 15.3136, 15.0064, 15.0683, 15.4724, 15.4670,
        14.9629, 15.0683, 15.1636, 14.3751, 15.6549, 15.5383, 15.1075, 15.3904,
        15.4737, 15.0170, 14.6375, 15.4467, 16.2500, 15.1636, 15.9003, 15.1452,
        14.8939, 14.9530, 15.2506, 15.3306, 14.8198, 15.4047, 15.8747, 15.1636,
        14.9957, 14.8939, 15.7258, 15.6336, 15.0064, 14.9607, 15.2253, 15.5244,
        15.5383, 15.2167, 15.2671, 15.5989, 15.2506, 15.2930, 16.0391, 15.6210,
        15.4161, 15.2589, 15.4047, 15.32

In [176]:
BATCH_SIZE = 8
from torch.utils.data import TensorDataset, DataLoader

train_ds = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
)
test_ds = DataLoader(
    TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
)
val_ds = DataLoader(
    TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
)

In [177]:
len(train_ds)

43

In [184]:
from typing import List, Any
import dataclasses


@dataclasses.dataclass
class TrainResult:
    # 1. Required Parameters (Must be passed at instantiation)
    num_epochs: int
    lr: float
    model_state: dict = dataclasses.field(default_factory=dict)

    # 2. Optional Selected Parameters (Automatically initialized if omitted)
    train_loss: List[float] = dataclasses.field(default_factory=list)
    train_acc: List[float] = dataclasses.field(default_factory=list)
    val_loss: List[float] = dataclasses.field(default_factory=list)
    val_acc: List[float] = dataclasses.field(default_factory=list)

    # Simple scalar defaults
    test_loss: float = 0.0
    test_acc: float = 0.0

In [191]:
# 1. Pre-allocate tracking dictionary or list outside the function
result = TrainResult(num_epochs=epochs, lr=lr)


def train_and_validate():
    model.train()

    # 2. Ensure target shape matches model output shape [N, 1]
    # This prevents catastrophic broadcasting errors in MSELoss
    # targets = y_train.unsqueeze(1) if y_train.dim() == 1 else y_train

    for epoch in range(10):
        # Forward pass
        for X_train, y_train in train_ds:
            y_hat = model(X_train)
            loss = loss_fn(y_hat, y_train.unsqueeze(1))

            # Record metrics using item() to extract raw float from GPU/CPU tensor
            result.train_loss.append(loss.item())

            # Backward pass
            loss.backward()
            optim.step()
            optim.zero_grad()
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for batch_X, batch_Y in val_ds:
                pred = model(batch_X)
                val_loss = loss_fn(pred, batch_Y.unsqueeze(1))
                result.val_loss.append(val_loss)

        # Optional: Print progress
        print(
            f"Epoch {epoch+1}/10 | Loss: {result.train_loss[-1]:.4f}, val_loss: {result.val_loss[-1]: .4f}"
        )

In [192]:
train_and_validate()

Epoch 1/10 | Loss: 0.0380, val_loss:  0.0602
Epoch 2/10 | Loss: 0.0475, val_loss:  0.0663
Epoch 3/10 | Loss: 0.2309, val_loss:  0.0535
Epoch 4/10 | Loss: 0.1967, val_loss:  0.0894
Epoch 5/10 | Loss: 0.0962, val_loss:  0.0279
Epoch 6/10 | Loss: 0.1286, val_loss:  0.0832
Epoch 7/10 | Loss: 0.0787, val_loss:  0.0452
Epoch 8/10 | Loss: 0.0584, val_loss:  0.0666
Epoch 9/10 | Loss: 0.0129, val_loss:  0.0661
Epoch 10/10 | Loss: 0.0942, val_loss:  0.0442


/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [193]:
result

TrainResult(num_epochs=10, lr=0.01, model_state={}, train_loss=[0.24641652405261993, 0.14191487431526184, 0.06658726930618286, 0.06387634575366974, 0.021539654582738876, 0.12308327108621597, 0.23540164530277252, 0.11109901964664459, 0.10209575295448303, 0.21929672360420227, 0.1265113353729248, 0.20311830937862396, 0.13847137987613678, 0.0695783719420433, 0.06719599664211273, 0.09957846254110336, 0.11950808018445969, 0.17635300755500793, 0.05779867619276047, 0.3081706166267395, 0.25097161531448364, 0.1118827760219574, 0.43861234188079834, 0.09145759046077728, 0.15352889895439148, 0.22250671684741974, 0.07044973969459534, 0.05779724940657616, 0.07002086937427521, 0.06394191086292267, 0.08409669995307922, 0.16975241899490356, 0.16404342651367188, 0.09676352143287659, 0.2802688181400299, 0.09992951154708862, 0.031719908118247986, 0.08739182353019714, 0.23081456124782562, 0.11204478144645691, 0.029810307547450066, 0.14057892560958862, 0.037975799292325974, 0.02431696653366089, 0.07346717268

In [ ]:
# with torch.inference_mode():